# PD02 · Consultar y resumir EVA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es traducir una pregunta a filtros verificables y resúmenes. La pregunta de trabajo es: ¿cómo se distribuyen los registros agrícolas de Boyacá entre cultivos durante 2023 y 2024?

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD02_consultas_eva.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD02_consultas_eva.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 5, 8, 10 y las recetas de porcentajes y equivalencia de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [3]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD02"
archivos = ['eva.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD02


## 1. Preparación de columnas · 10 minutos

Inspeccionar la tabla y crear `codigo_departamento` a partir de `Código Dane departamento`, `anio` desde `Año`, `area_ha` desde `Área cosechada` y `produccion_t` desde `Producción`. El código se conserva como texto; año, área y producción se convierten a tipos numéricos. Registrar las ausencias resultantes.

📘 **Apoyo:** `.str.strip`, `pd.to_numeric`, `astype("Int64")` e `isna`.

✅ **Comprobación:** Las columnas originales se conservan y no se eliminan filas. El corte de entrada contiene 166.732 registros.

In [14]:
# Desarrollo de la actividad 1
# Preparar las cuatro columnas de trabajo en eva.

# Código del departamento: conservar como texto
eva["codigo_departamento"] = (
    eva["Código Dane departamento"]
    .astype("string")
    .str.strip()
)

# Año: convertir a número entero que admite valores ausentes
eva["anio"] = pd.to_numeric(
    eva["Año"].astype("string").str.strip(),
    errors="coerce"
).astype("Int64")

# Área cosechada: convertir a número
eva["area_ha"] = pd.to_numeric(
    eva["Área cosechada"].astype("string").str.strip(),
    errors="coerce"
)

# Producción: convertir a número
eva["produccion_t"] = pd.to_numeric(
    eva["Producción"].astype("string").str.strip(),
    errors="coerce"
)

print("Dimensiones del DataFrame:", eva.shape)

# Ausencias
ausencias = pd.DataFrame({
    "columna": [
        "codigo_departamento",
        "anio",
        "area_ha",
        "produccion_t"
    ],
    "cantidad_ausente": [
        eva["codigo_departamento"].isna().sum(),
        eva["anio"].isna().sum(),
        eva["area_ha"].isna().sum(),
        eva["produccion_t"].isna().sum()
    ]
})

print(ausencias)

# Verificar que no se eliminaron filas
print("Total de registros:", len(eva))
print("¿Se conservan las 166.732 filas?:", len(eva) == 166732)

# Revisar las columnas nuevas y sus tipos
print(eva[[
    "codigo_departamento",
    "anio",
    "area_ha",
    "produccion_t"
]].dtypes)

# Comparar algunas filas con los datos originales
eva[[
    "Código Dane departamento",
    "codigo_departamento",
    "Año",
    "anio",
    "Área cosechada",
    "area_ha",
    "Producción",
    "produccion_t"
]].head()

# Mostrar los tipos y los valores que no pudieron convertirse.
print("Tipos de datos:")
print(eva[[
    "codigo_departamento",
    "anio",
    "area_ha",
    "produccion_t"
]].dtypes)

# Valores que no pudieron convertirse
columnas_originales = {
    "anio": "Año",
    "area_ha": "Área cosechada",
    "produccion_t": "Producción"
}

print("\nValores que no pudieron convertirse:")

for columna_nueva, columna_original in columnas_originales.items():
    original = eva[columna_original].astype("string").str.strip()
    convertida = eva[columna_nueva]

    # Valores originales no vacíos que terminaron como ausentes
    fallidos = original.ne("") & convertida.isna()

    print(f"\n{columna_original}:")
    print(eva.loc[fallidos, columna_original].value_counts())

Dimensiones del DataFrame: (166732, 22)
               columna  cantidad_ausente
0  codigo_departamento                 0
1                 anio                 0
2              area_ha                 0
3         produccion_t                 0
Total de registros: 166732
¿Se conservan las 166.732 filas?: True
codigo_departamento    string[python]
anio                            Int64
area_ha                       Float64
produccion_t                  Float64
dtype: object
Tipos de datos:
codigo_departamento    string[python]
anio                            Int64
area_ha                       Float64
produccion_t                  Float64
dtype: object

Valores que no pudieron convertirse:

Año:
Series([], Name: count, dtype: int64)

Área cosechada:
Series([], Name: count, dtype: int64)

Producción:
Series([], Name: count, dtype: int64)


**Interpretación del resultado:**

La conversión de las columnas de año, área cosechada y producción se realizó sin identificar valores no vacíos que no pudieran convertirse a números.

## 2. Selección con máscara y loc · 15 minutos

Seleccionar el departamento `15` (Boyacá), los años 2023 y 2024, área cosechada positiva y producción no negativa. Guardar todas las columnas de las filas seleccionadas en `consulta_loc`, utilizando `.copy()`. Presentar el total original, el seleccionado y la diferencia.

📘 **Apoyo:** `.eq`, `.isin`, `.gt`, `.ge`, `&` y `.loc`.

✅ **Comprobación:** Todas las filas seleccionadas cumplen simultáneamente las cuatro condiciones. La diferencia respecto a la entrada incluye registros fuera del alcance territorial o temporal; no implica que todos sean errores.

In [18]:
# Desarrollo de la actividad 2
# Construir la máscara y guardar consulta_loc.

mascara = (
    eva["codigo_departamento"].eq("15") &
    eva["anio"].isin([2023, 2024]) &
    eva["area_ha"].gt(0) &
    eva["produccion_t"].ge(0)
)

# Guardar todas las columnas de las filas seleccionadas
consulta_loc = eva.loc[mascara].copy()

# Mostrar los conteos
total_original = len(eva)
total_seleccionado = len(consulta_loc)
diferencia = total_original - total_seleccionado

print("Total original:", total_original)
print("Total seleccionado:", total_seleccionado)
print("Diferencia:", diferencia)

Total original: 166732
Total seleccionado: 4924
Diferencia: 161808


**Interpretación del resultado:**

La diferencia representa los registros que no fueron seleccionados porque no cumplen al menos una de las condiciones territoriales, temporales o numéricas. No significa necesariamente que sean registros erróneos.

## 3. Consulta equivalente con query · 10 minutos

Repetir exactamente la selección anterior mediante `.query`, con `engine="python"`, y guardar el resultado en `consulta_query`. Usar una variable para el código departamental y referenciarla con `@`. Comparar ambos DataFrames sin eliminar columnas.

📘 **Apoyo:** `.query`, `@variable` y `pd.testing.assert_frame_equal`.

✅ **Comprobación:** La comparación debe confirmar las mismas filas, columnas y valores. Si falla, se revisan las condiciones y los tipos antes de continuar.

In [21]:
# Desarrollo de la actividad 3
# Construir consulta_query.

# Repetir la selección mediante query

codigo_depto = "15"

consulta_query = eva.query(
    'codigo_departamento == @codigo_depto and '
    'anio in [2023, 2024] and '
    'area_ha > 0 and '
    'produccion_t >= 0',
    engine="python"
).copy()

print("Total seleccionado con query:", len(consulta_query))


# Comprobar que consulta_loc y consulta_query son equivalentes.

pd.testing.assert_frame_equal(
    consulta_loc,
    consulta_query
)

print("Ambos DataFrames son equivalentes.")

Total seleccionado con query: 4924
Ambos DataFrames son equivalentes.


**Interpretación del resultado:**

La comparación con pd.testing.assert_frame_equal() confirmó que ambos DataFrames contienen las mismas filas, columnas, valores y tipos de datos. Esto demuestra que los dos métodos permiten aplicar correctamente las cuatro condiciones.

## 4. Conteos y porcentajes · 20 minutos

Sobre `consulta_loc`, construir `resumen` con una fila por año y cultivo y una columna `registros`. Añadir el total de registros de cada año y `porcentaje_anual`. Ordenar los resultados por año y cantidad de registros descendente. Conservar los grupos con claves ausentes mediante `dropna=False`.

📘 **Apoyo:** `groupby`, `size`, `reset_index`, `transform("sum")` y `sort_values`.

✅ **Comprobación:** La suma de registros coincide con el tamaño de consulta_loc. Los porcentajes suman aproximadamente 100 % por año; se admite una pequeña tolerancia numérica y se redondea solo para mostrar.

In [23]:
# Desarrollo de la actividad 4
# Crear resumen con registros, total_anio y porcentaje_anual.
# Agrupar por año y cultivo y contar registros
resumen = (
    consulta_loc
    .groupby(["anio", "Cultivo"], dropna=False)
    .size()
    .reset_index(name="registros")
)

# Calcular el total de registros por año
resumen["total_anual"] = (
    resumen.groupby("anio", dropna=False)["registros"]
    .transform("sum")
)

# Calcular el porcentaje de cada cultivo dentro de su año
resumen["porcentaje_anual"] = (
    resumen["registros"] / resumen["total_anual"] * 100
)

# Ordenar por año y cantidad de registros descendente
resumen = resumen.sort_values(
    ["anio", "registros"],
    ascending=[True, False]
).reset_index(drop=True)

print(resumen)

# Verificar los totales y los porcentajes por año.

print("Registros en consulta_loc:", len(consulta_loc))
print("Suma de registros en resumen:", resumen["registros"].sum())

# Porcentajes
print("\nPorcentajes por año:")
print(
    resumen.groupby("anio", dropna=False)["porcentaje_anual"]
    .sum()
    .round(6)
)

     anio                                            Cultivo  registros  \
0    2023                                               Papa        239   
1    2023                                               Maíz        194   
2    2023                                             Arveja        176   
3    2023                                             Frijol        162   
4    2023                                             Tomate        113   
..    ...                                                ...        ...   
187  2024                                              Dátil          1   
188  2024                                          Macadamia          1   
189  2024                                              Olivo          1   
190  2024  Otros cultivos para condimentos, bebidas medic...          1   
191  2024                                               Piña          1   

     total_anual  porcentaje_anual  
0           2436          9.811166  
1           2436         

**Interpretación del resultado:**

El resumen contiene 192 combinaciones de año y cultivo, correspondientes a un total de 4.924 registros seleccionados para el departamento de Boyacá, en los años 2023 y 2024, que cumplen las condiciones establecidas de área cosechada y producción.

La suma de la columna registros es igual a 4.924, que coincide con el total de filas de consulta_loc. Esto confirma que el agrupamiento conserva todos los registros seleccionados.

## 5. Tabla dinámica y conclusiones · 15 minutos

Construir una tabla dinámica a partir de `resumen`, con cultivos en filas, años en columnas y conteos como valores. Completar con cero las combinaciones sin registros. Exportar `resumen` a `OUT / "resumen_eva.csv"`, sin índice, y redactar dos conclusiones.

📘 **Apoyo:** `pivot_table`, `fill_value=0` y `to_csv`.

✅ **Comprobación:** Los totales de la tabla dinámica concilian con resumen. Las conclusiones se refieren a la cantidad de registros y no a la participación de cada cultivo en la superficie agrícola.

In [26]:
# Desarrollo de la actividad 5
# Construir y mostrar la tabla dinámica.
tabla_dinamica = resumen.pivot_table(
    index="Cultivo",
    columns="anio",
    values="registros",
    aggfunc="sum",
    fill_value=0,
    dropna=False
)

print(tabla_dinamica)

# Verificar que los totales coinciden
print("Total en resumen:", resumen["registros"].sum())
print("Total en tabla dinámica:", tabla_dinamica.sum().sum())

print("\nTotales por año en la tabla dinámica:")
print(tabla_dinamica.sum(axis=0))

# Exportar resumen y explicar dos resultados en la celda siguiente.
from pathlib import Path

OUT = Path("OUT")
OUT.mkdir(parents=True, exist_ok=True)

resumen.to_csv(
    OUT / "resumen_eva.csv",
    index=False
)

print("Archivo guardado en:", OUT / "resumen_eva.csv")

anio             2023  2024
Cultivo                    
Acelga              5     8
Achira              4     4
Agraz - mortiño     3     4
Aguacate           48    52
Ahuyama            23    26
...               ...   ...
Uchuva             25    21
Uva                 6     6
Yuca               82    82
Zanahoria          44    46
Zapote              5     6

[97 rows x 2 columns]
Total en resumen: 4924
Total en tabla dinámica: 4924

Totales por año en la tabla dinámica:
anio
2023    2436
2024    2488
dtype: int64
Archivo guardado en: OUT/resumen_eva.csv


**Interpretación del resultado:**

1. El análisis agrupó un total de **4.924 registros**, distribuidos entre 2023 y 2024. En 2023 hubo 2.436 registros y en 2024, 2.488, es decir, 52 registros más en 2024. Esto muestra una mayor cantidad de observaciones registradas en el segundo año.

2. La tabla dinámica permite comparar la cantidad de registros de cada cultivo entre 2023 y 2024. Las combinaciones sin registros se completaron con cero, para facilitar la comparación entre años.

## Entrega

El notebook completado se conserva como `soluciones/PD02_consultas_eva.ipynb`. La salida regenerable es `kit/salidas/soluciones/PD02/resumen_eva.csv`.

✅ Deben quedar visibles la equivalencia de consultas, la conciliación de los conteos, los porcentajes por año, la tabla dinámica y dos conclusiones.

⚠️ Una fila EVA es una observación de la fuente, no un predio ni una persona. Un porcentaje de registros no equivale a un porcentaje de área cultivada. El cero de una tabla de conteos indica que no se encontraron filas para esa combinación.